In [ ]:
import pandas as pd
import os
from pathlib import Path
import re
import numpy as np # Needed for NaN checks

# ==============================================
# Configuration (Ensure these match your data)
# ==============================================
# (PARENT_DATA_DIR, OUTPUT_DATA_DIR, OUTPUT_FILENAME, CSV_FILENAME, RESULTS_SUBFOLDER as before)
PARENT_DATA_DIR = Path.cwd() # Or Path("C:/path/to/folder_containing_ferm_xx")
SCRIPT_DIR = Path.cwd() # Assuming script is run from where output 'data' should be relative to
OUTPUT_DATA_DIR = SCRIPT_DIR / "data"
OUTPUT_FILENAME = "traintest_compiled.xlsx"
CSV_FILENAME = "processed_data.csv"
RESULTS_SUBFOLDER = "results"


# --- Column Names ---
# Column names as written by the current pipeline (analyze_doe_run.py saves
# the standard internal names). Updated 2026-10-23: the refactored pipeline
# saves standard 'X [unit]' column names instead of the old snake_case ones.
TIME_COL = 'Process time [h]'
PHASE_COL = 'Process phase [-]'
BIOMASS_COL = 'Biomass [g/L]'
PRODUCT_COL = 'Protein [g/L]'
# Add other columns you decided to keep:
TEMP_COL = 'Temperature [°C]'
FEED_COL = 'Feed [mL]'
VOLUME_COL = 'Volume corrected [mL]'
INDUCED_COL = 'Is induced [-]'
PH_COL = 'pH [-]'
PO2_COL = 'pO2 [%]'
STIRRER_COL = 'Stirrer speed [1/min]'
# Define the internal names you want after renaming
COLUMNS_TO_KEEP_AND_RENAME = {
    TIME_COL: 'time',
    BIOMASS_COL: 'X',
    PRODUCT_COL: 'P',
    TEMP_COL: 'temp',
    FEED_COL: 'feed',
    VOLUME_COL: 'reactor_volume',
    INDUCED_COL: 'inductor_switch',
    PH_COL: 'pH',
    PO2_COL: 'pO2',
    STIRRER_COL: 'StirrerSpeed',
    # Add any other keepers here, e.g.:
    'Offgas O2 [%]': 'OffgasO2',
    'Offgas CO2 [%]': 'OffgasCO2',
    'Glucose [g/L]': 'Glucose',
}

# --- Phase Names ---
BATCH_PHASE_NAME = "Batch"
FEDBATCH_PHASE_NAME = "Fedbatch" # Case-sensitive match needed

# ==============================================
# Script Logic
# ==============================================

def process_single_run(df_run: pd.DataFrame, run_id: str) -> pd.DataFrame:
    """
    Processes data for a single run: filters batch, finds last batch measurement,
    and imputes it into the first fedbatch row. Selects and renames columns.

    Args:
        df_run: DataFrame containing data for one specific run.
        run_id: The identifier for the run.

    Returns:
        Processed DataFrame for the run, or empty DataFrame if processing fails.
    """
    print(f"    Processing RunID: {run_id}...")

    # Ensure time column is sorted
    if TIME_COL not in df_run.columns:
        print(f"    Error: Time column '{TIME_COL}' not found for RunID {run_id}. Skipping.")
        return pd.DataFrame()
    df_run = df_run.sort_values(by=TIME_COL).reset_index(drop=True)

    # --- 1. Find Last Batch Measurements & Transition Time ---
    last_batch_biomass = np.nan
    last_batch_product = np.nan
    t_feed_start = np.nan

    # Find the index of the *last* row belonging to the BATCH phase
    batch_rows = df_run[df_run[PHASE_COL] == BATCH_PHASE_NAME]
    if not batch_rows.empty:
        last_batch_idx = batch_rows.index[-1]

        # Find the last non-NaN biomass measurement *within or at the end of* the batch phase
        biomass_in_batch = batch_rows[BIOMASS_COL].dropna()
        if not biomass_in_batch.empty:
            last_batch_biomass = biomass_in_batch.iloc[-1]
            print(f"      Found last batch biomass measurement: {last_batch_biomass:.3f} at index {biomass_in_batch.index[-1]}")
        else:
             print(f"      Warning: No non-NaN biomass measurements found in batch phase for RunID {run_id}.")

        # Find the last non-NaN product measurement *within or at the end of* the batch phase
        product_in_batch = batch_rows[PRODUCT_COL].dropna()
        if not product_in_batch.empty:
            last_batch_product = product_in_batch.iloc[-1]
            print(f"      Found last batch product measurement: {last_batch_product:.3f} at index {product_in_batch.index[-1]}")
        else:
            print(f"      No non-NaN product measurements found in batch phase for RunID {run_id}. Assuming 0.")
            last_batch_product = 0.0 # Assume product is 0 if not measured in batch

        # Find the first row *after* the last batch row that is 'Fedbatch'
        fedbatch_start_candidates = df_run.loc[last_batch_idx + 1:]
        first_fedbatch_row = fedbatch_start_candidates[fedbatch_start_candidates[PHASE_COL] == FEDBATCH_PHASE_NAME].first_valid_index()

        if first_fedbatch_row is not None:
            t_feed_start = df_run.loc[first_fedbatch_row, TIME_COL]
            print(f"      Determined Fedbatch start time (t_feed_start): {t_feed_start:.3f} at index {first_fedbatch_row}")
        else:
            print(f"    Error: Could not find start of '{FEDBATCH_PHASE_NAME}' phase after Batch for RunID {run_id}. Skipping run.")
            return pd.DataFrame()
    else:
        # Handle cases with no "Batch" phase - maybe start from time 0? Or first Fedbatch?
        print(f"    Warning: No rows found with phase '{BATCH_PHASE_NAME}' for RunID {run_id}.")
        # Attempt to find the first Fedbatch row from the start
        first_fedbatch_row = df_run[df_run[PHASE_COL] == FEDBATCH_PHASE_NAME].first_valid_index()
        if first_fedbatch_row is not None:
            t_feed_start = df_run.loc[first_fedbatch_row, TIME_COL]
            print(f"      Starting from first '{FEDBATCH_PHASE_NAME}' row at time {t_feed_start:.3f}.")
            # Assume initial X and P are the values in this first row, or NaN if missing
            last_batch_biomass = df_run.loc[first_fedbatch_row, BIOMASS_COL]
            last_batch_product = df_run.loc[first_fedbatch_row, PRODUCT_COL]
            if pd.isna(last_batch_product): last_batch_product = 0.0 # Default P to 0
        else:
             print(f"    Error: Could not find any '{FEDBATCH_PHASE_NAME}' phase for RunID {run_id}. Skipping run.")
             return pd.DataFrame()


    # --- 2. Filter Data to Keep Fedbatch Onwards ---
    # Keep all rows from t_feed_start onwards
    df_processed = df_run[df_run[TIME_COL] >= t_feed_start].copy()
    if df_processed.empty:
        print(f"    Error: No data remaining after filtering for time >= {t_feed_start:.3f} for RunID {run_id}. Skipping run.")
        return pd.DataFrame()
    print(f"      Filtered data starting from t={t_feed_start:.3f}. Shape: {df_processed.shape}")


    # --- 3. Impute First Fedbatch Row with Last Batch Measurements ---
    first_fb_index = df_processed.index[0]
    if pd.isna(df_processed.loc[first_fb_index, BIOMASS_COL]):
        if not pd.isna(last_batch_biomass):
            df_processed.loc[first_fb_index, BIOMASS_COL] = last_batch_biomass
            print(f"      Imputed biomass {last_batch_biomass:.3f} into first fedbatch row (index {first_fb_index}).")
        else:
            # No batch-phase biomass measurement exists (first sample drawn
            # after feed start; true for ferm_08/09/10/12/ferm_opt on the
            # corrected time base). Backfill the first available fedbatch
            # biomass so X has an initial value at the solver's start time
            # (updated 2026-10-24; the old pipeline had stamped that fedbatch
            # sample as 'last batch biomass', which was a mislabel).
            fedbatch_biomass = df_processed[BIOMASS_COL].dropna()
            if not fedbatch_biomass.empty:
                src_idx = fedbatch_biomass.index[0]
                df_processed.loc[first_fb_index, BIOMASS_COL] = fedbatch_biomass.iloc[0]
                print(
                    f"      Backfilled biomass {fedbatch_biomass.iloc[0]:.3f} from t="
                    f"{df_processed.loc[src_idx, TIME_COL]:.3f} into first fedbatch row (t="
                    f"{df_processed.loc[first_fb_index, TIME_COL]:.3f})."
                )
            else:
                print(f"      Warning: First fedbatch biomass is NaN and no biomass measurement found anywhere for RunID {run_id}.")
    #else: biomass already exists in first fb row

    if pd.isna(df_processed.loc[first_fb_index, PRODUCT_COL]):
        # Use last batch measurement or the default 0.0 if none found
        impute_p = last_batch_product if not pd.isna(last_batch_product) else 0.0
        df_processed.loc[first_fb_index, PRODUCT_COL] = impute_p
        print(f"      Imputed product {impute_p:.3f} into first fedbatch row (index {first_fb_index}).")
    #else: product already exists in first fb row


    # --- 4. Select and Rename Columns ---
    columns_to_actually_keep = [col for col in COLUMNS_TO_KEEP_AND_RENAME.keys() if col in df_processed.columns]
    if not columns_to_actually_keep:
        print(f"    Error: None of the desired columns found in the processed data for RunID {run_id}. Skipping run.")
        return pd.DataFrame()

    df_final = df_processed[columns_to_actually_keep].copy()
    df_final = df_final.rename(columns=COLUMNS_TO_KEEP_AND_RENAME)
    print(f"      Selected and renamed columns. Final shape for run: {df_final.shape}")

    # --- 5. Add RunID ---
    df_final['RunID'] = run_id

    return df_final


def find_and_compile_data_processed(parent_dir: Path, csv_name: str, results_folder: str) -> pd.DataFrame:
    """
    Finds processed data CSVs, processes each run individually (filter, impute),
    and concatenates the results. Specifies semicolon delimiter.
    """
    all_processed_frames = []
    found_files_count = 0

    print(f"Searching for '{csv_name}' in '{results_folder}' within subfolders of: {parent_dir}")

    for item in sorted(parent_dir.iterdir()):
        if item.is_dir() and item.name.lower().startswith('ferm_'):
            run_id = item.name
            results_path = item / results_folder
            csv_file_path = results_path / csv_name

            if csv_file_path.is_file():
                print(f"  Found: {csv_file_path}")
                try:
                    # Read the CSV using semicolon delimiter
                    df_raw_run = pd.read_csv(csv_file_path, delimiter=';')

                    if df_raw_run.shape[1] <= 1:
                         print(f"    Warning: File {csv_file_path} was read with only {df_raw_run.shape[1]} column(s). Check delimiter. Skipping.")
                         continue

                    # Process this single run's data
                    df_processed_run = process_single_run(df_raw_run, run_id)

                    if not df_processed_run.empty:
                        all_processed_frames.append(df_processed_run)
                        found_files_count += 1

                except Exception as e:
                    print(f"    Error reading or processing file {csv_file_path} for RunID {run_id}: {e}")
                    import traceback
                    traceback.print_exc() # Print full traceback for debugging
            else:
                print(f"  Skipping: {item.name} (File not found: {csv_file_path})")

    if not all_processed_frames:
        print("No data files found or successfully processed.")
        return pd.DataFrame()

    print(f"\nConcatenating processed data from {found_files_count} runs...")
    combined_df = pd.concat(all_processed_frames, ignore_index=True)
    print(f"Combined DataFrame shape: {combined_df.shape}")
    print(f"Columns in combined DataFrame: {combined_df.columns.tolist()}")
    print(f"Unique RunIDs found: {combined_df['RunID'].unique().tolist()}")

    return combined_df

# (save_compiled_data function remains the same)
def save_compiled_data(df: pd.DataFrame, output_dir: Path, filename: str):
    """Saves the DataFrame to the specified directory and filename."""
    if df.empty:
        print("DataFrame is empty. Nothing to save.")
        return
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    try:
        if filename.endswith(".xlsx"):
            df.to_excel(output_path, index=False, engine='openpyxl')
            print(f"Successfully saved compiled data to Excel: {output_path}")
            print("Note: Ensure 'openpyxl' is installed (`pip install openpyxl`).")
        else: # Default to CSV if not .xlsx
           if not filename.endswith(".csv"):
                print(f"Warning: Unknown output file extension for '{filename}'. Saving as CSV.")
                output_path = output_path.with_suffix('.csv')
           df.to_csv(output_path, index=False)
           print(f"Successfully saved compiled data to CSV: {output_path}")
    except Exception as e:
        print(f"Error saving data to {output_path}: {e}")


# ==============================================
# Main Execution
# ==============================================
if __name__ == "__main__":
    print("--- Starting Data Compilation Script ---")

    if not PARENT_DATA_DIR.is_dir():
        print(f"\nError: Parent data directory does not exist: '{PARENT_DATA_DIR}'")
        print("Please adjust the PARENT_DATA_DIR variable.")
    else:
        # Use the new processing function
        compiled_dataframe = find_and_compile_data_processed(
            parent_dir=PARENT_DATA_DIR,
            csv_name=CSV_FILENAME,
            results_folder=RESULTS_SUBFOLDER
        )

        save_compiled_data(
            df=compiled_dataframe,
            output_dir=OUTPUT_DATA_DIR,
            filename=OUTPUT_FILENAME
        )

    print("\n--- Data Compilation Script Finished ---")